# TP : du sinogramme à la mesure

**Fil rouge** : objet → projection → bruit → reconstruction → mesure. Tout est simulé : vous jouez le rôle de la machine.

| Partie | Thème | Durée indicative | Lien avec le cours |
|---|---|---|---|
| 0 | Prise en main | 10 min | |
| 1 | Projection et sinogramme | 25 min | séance 3, partie II |
| 2 | Rétroprojection simple et filtrée | 35 min | séance 3, partie II |
| | *Pause* | 10 min | |
| 3 | Bruit statistique | 25 min | séances 2 et 3, partie III |
| 4 | Reconstruction itérative MLEM | 35 min | séance 3, partie IV |
| 5 | Résolution et volume partiel | 30 min | séance 2 (PSF, RC) |
| ⭐ | Bonus, pour ceux qui ont fini | — | |

**Aucune compétence en programmation n'est demandée.** Vous n'aurez jamais à écrire une fonction :
seulement à remplacer des `...` par un nombre, un nom, ou une ligne recopiée et modifiée. Le travail, ce sont les **questions**.

## Mode d'emploi

- Une cellule s'exécute avec **Maj + Entrée**. Exécutez les cellules **dans l'ordre**, de haut en bas.
- ✏️ **À compléter** : remplacez chaque `...` comme indiqué dans le commentaire, puis exécutez.
- 🔍 **Questions** : répondez dans la cellule « ✍️ Votre réponse » juste en dessous (double-clic pour écrire).
- ✅ **Checkpoint** : montrez le résultat à l'enseignant avant de continuer.
- En cas d'erreur incompréhensible : menu *Kernel → Restart*, puis réexécutez les cellules depuis le début. Si cela persiste, appelez l'enseignant.
- Le message `NameError: name '...' is not defined` signifie presque toujours qu'une cellule précédente n'a pas été exécutée.

## 0. Prise en main (à exécuter en premier)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.data import shepp_logan_phantom
from skimage.transform import radon, iradon, resize
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
from ipywidgets import interact, IntSlider
import warnings; warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (10, 4)
rng = np.random.default_rng(2026)

N = 128                                   # image N x N pixels
PIX = 2.0                                 # taille du pixel (mm) : champ de 256 mm
THETA = np.linspace(0., 180., N, endpoint=False)   # angles de projection (degrés)

def disque(cx, cy, r, val=1.0, n=N):
    """Image n x n contenant un disque de rayon r (pixels) centré en (cx, cy) : cx = colonne, cy = ligne."""
    y, x = np.mgrid[:n, :n]
    return val * (((x - cx)**2 + (y - cy)**2) <= r**2)

def montrer(*imgs, titres=None, cmap="gray"):
    """Affiche plusieurs images côte à côte."""
    fig, ax = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4))
    ax = np.atleast_1d(ax)
    for i, im in enumerate(imgs):
        ax[i].imshow(im, cmap=cmap, aspect="auto" if im.shape[0] != im.shape[1] else "equal")
        ax[i].axis("off")
        if titres:
            ax[i].set_title(titres[i])
    plt.show()

print("Environnement prêt.")

### Exercice 0 : premier contact
✏️ Affichez un disque de rayon **20 pixels**, centré en (64, 64). Puis changez le centre et le rayon, et réexécutez.

In [ ]:
mon_disque = disque(64, 64, ...)        # ✏️ remplacez ... par le rayon (en pixels)
montrer(mon_disque, titres=["Mon disque"])

---
# Partie 1 : Projection et sinogramme

Un détecteur ne voit pas l'objet : il mesure, sous chaque angle $\theta$, la **somme** de l'activité le long de chaque ligne
(en TEP, le long de chaque ligne de réponse entre deux détecteurs). L'ensemble de ces projections, rangées angle par angle,
forme le **sinogramme**. La fonction `radon` fait ce calcul.

In [ ]:
objet = resize(shepp_logan_phantom(), (N, N))
sino = radon(objet, theta=THETA)
montrer(objet, sino, titres=["Objet", "Sinogramme (position verticale, angle horizontal)"])

### Exercice 1.1 : le sinogramme d'un point
✏️ Créez une source ponctuelle (disque de rayon 1 pixel) **décentrée**, en (90, 64), puis calculez son sinogramme
en vous inspirant de la cellule précédente.

In [ ]:
point = disque(..., ..., 1)               # ✏️ centre (90, 64)
sino_point = radon(..., theta=THETA)      # ✏️ quelle image projette-t-on ?
montrer(point, sino_point, titres=["Source ponctuelle", "Sinogramme"])

Déplacez maintenant le point avec les curseurs (aucun code à écrire).

In [ ]:
@interact(colonne=IntSlider(min=20, max=108, value=90), ligne=IntSlider(min=20, max=108, value=64))
def deplacer(colonne, ligne):
    p = disque(colonne, ligne, 1)
    montrer(p, radon(p, theta=THETA), titres=[f"point en ({colonne}, {ligne})", "sinogramme"])

🔍 **Questions**
1. Quelle forme a la trace ? D'où vient le nom « sinogramme » ?
2. Qu'est-ce qui fixe l'**amplitude** de la courbe ? Sa **phase** (la position de son maximum) ?
3. Pourquoi 180° suffisent-ils ici ? (pensez aux deux photons de 511 keV émis dos à dos en TEP)

✍️ **Votre réponse :**



### ✅ Checkpoint 1
✏️ Affichez côte à côte les sinogrammes de deux points : l'un au **centre** (64, 64), l'autre près du **bord** (110, 64).

In [ ]:
p_centre = disque(..., ..., 1)            # ✏️
p_bord   = disque(..., ..., 1)            # ✏️
montrer(radon(p_centre, theta=THETA), radon(p_bord, theta=THETA), titres=["point au centre", "point au bord"])

---
# Partie 2 : Rétroprojection

**Idée naïve** : redistribuer chaque projection uniformément le long de la ligne d'où elle vient, et additionner sur tous les angles.
C'est `iradon` avec `filter_name=None`.

In [ ]:
bp = iradon(sino, theta=THETA, filter_name=None)
montrer(objet, bp, titres=["Objet", "Rétroprojection simple"])

### Exercice 2.1 : le flou en étoile
✏️ Rétroprojetez (sans filtre) le sinogramme d'une source ponctuelle **centrée**. Le profil horizontal est tracé
en échelle logarithmique, avec en pointillés une courbe en $1/r$ pour comparaison.

In [ ]:
sino_c = radon(disque(64, 64, 1), theta=THETA)
bp_c = iradon(sino_c, theta=THETA, filter_name=...)      # ✏️ sans filtre : quelle valeur ? (voir la cellule précédente)

r = np.abs(np.arange(N) - 64) + 0.5
plt.plot(bp_c[64], label="profil de la rétroprojection")
plt.plot(bp_c[64].max() * 0.5 / r, "k--", label="courbe en 1/r")
plt.yscale("log"); plt.xlabel("pixel"); plt.legend(); plt.title("Profil ligne 64"); plt.show()

🔍 **Question** : comment décroissent les « queues » du profil ? Pourquoi l'image est-elle floue même sans bruit
ni imperfection du détecteur ?

✍️ **Votre réponse :**



### Exercice 2.2 : la rétroprojection filtrée (FBP)
✏️ Comparez les filtres `"ramp"`, `"shepp-logan"` et `"hann"`. Il suffit de compléter **une** ligne :
on reconstruit `sino` avec le filtre `f`.

In [ ]:
filtres = ["ramp", "shepp-logan", "hann"]
recs = []
for f in filtres:
    recs.append( iradon(sino, theta=THETA, filter_name=...) )    # ✏️ quel filtre ? (la variable de la boucle)
montrer(*recs, titres=filtres)

for f, rr in zip(filtres, recs):
    plt.plot(rr[64], label=f)
plt.plot(objet[64], "k--", label="vrai"); plt.legend(); plt.title("Profil ligne 64"); plt.show()

🔍 **Questions**
1. Que fait le filtre rampe dans l'espace des fréquences ? Pourquoi corrige-t-il le flou ?
2. Classez les filtres du plus net au plus lisse. Quel est le prix de la netteté ? (gardez la réponse en tête pour la partie 3)

✍️ **Votre réponse :**



### Exercice 2.3 : échantillonnage angulaire
Reconstruisez (FBP rampe) avec 8, 16, 32 puis 128 angles. ✏️ Complétez la ligne de projection.

In [ ]:
recs_ang = []
for n_ang in [8, 16, 32, 128]:
    th = np.linspace(0., 180., n_ang, endpoint=False)          # n_ang angles régulièrement répartis
    s = radon(objet, theta=...)                                 # ✏️ projeter avec les angles th
    recs_ang.append(iradon(s, theta=th, filter_name="ramp"))
montrer(*recs_ang, titres=["8 angles", "16", "32", "128 angles"])

🔍 **Question** : quel artefact apparaît ? Quel lien avec le nombre de pas d'une gamma-caméra en TEMP ?

✍️ **Votre réponse :**



### ✅ Checkpoint 2
Montrez le graphe des profils de l'exercice 2.2 et dites quel filtre vous choisiriez pour une image clinique, et pourquoi.

---
☕ **Pause (10 min)**

---
# Partie 3 : Bruit statistique

En médecine nucléaire, on **compte** des photons : chaque case du sinogramme suit une loi de Poisson.
La fonction ci-dessous simule une acquisition réaliste, avec :
- le **flou du détecteur** (gaussienne de largeur `fwhm_mm` le long des projections) ;
- un nombre total de **coups** fixé ;
- le **bruit de Poisson**.

Rien à compléter : exécutez.

In [ ]:
def acquerir(objet, coups, fwhm_mm=0.0, theta=THETA):
    """Simule un sinogramme mesuré : projection, flou du détecteur, nombre de coups, bruit de Poisson."""
    p = radon(objet, theta=theta)
    if fwhm_mm > 0:
        p = gaussian_filter1d(p, fwhm_mm / PIX / 2.355, axis=0)
    p = np.clip(p, 0, None)
    p = p / p.sum() * coups
    return rng.poisson(p).astype(float)

cylindre = disque(64, 64, 50)            # fantôme uniforme de 20 cm de diamètre
roi_c = disque(64, 64, 20) > 0           # région d'intérêt centrale (vrai / faux pour chaque pixel)

montrer(*[acquerir(cylindre, c) for c in [1e4, 1e5, 1e6]], titres=["sinogramme, 10⁴ coups", "10⁵ coups", "10⁶ coups"])

### Exercice 3.1 : bruit et nombre de coups
Pour $10^5$, $10^6$ et $10^7$ coups, on reconstruit le cylindre (FBP rampe) et on calcule dans `roi_c` le
**coefficient de variation** : CV = écart-type / moyenne.

En Python, l'écart-type des pixels de l'image `r` dans la région `roi_c` s'écrit `r[roi_c].std()`, et leur moyenne `r[roi_c].mean()`.

✏️ Complétez les deux lignes.

In [ ]:
niveaux = [1e5, 1e6, 1e7]
imgs, cvs = [], []
for c in niveaux:
    y = acquerir(cylindre, c)
    r = iradon(y, theta=THETA, filter_name=...)                 # ✏️ FBP avec le filtre rampe
    cv = ...                                                    # ✏️ écart-type / moyenne dans roi_c
    imgs.append(r); cvs.append(cv)
    print(f"{c:.0e} coups : CV = {100*cv:.1f} %")
montrer(*imgs, titres=[f"{c:.0e} coups" for c in niveaux])

🔍 **Questions**
1. Par combien le CV est-il divisé quand on multiplie les coups par 10 ? Était-ce prévisible ?
2. Concrètement, comment augmenter le nombre de coups en clinique ? Avec quelles limites ?

✍️ **Votre réponse :**



### ✅ Checkpoint 3 : rampe ou Hann ?
La cellule trace le CV en fonction du nombre de coups (échelles log-log). ✏️ Ajoutez le filtre `"hann"` à la liste, puis exécutez.

In [ ]:
FILTRES = ["ramp", ...]                                         # ✏️ ajoutez "hann"
coups_liste = [1e5, 3e5, 1e6, 3e6, 1e7]
for f in FILTRES:
    cv_f = []
    for c in coups_liste:
        r = iradon(acquerir(cylindre, c), theta=THETA, filter_name=f)
        cv_f.append(r[roi_c].std() / r[roi_c].mean())
    plt.loglog(coups_liste, cv_f, "o-", label=f)
plt.loglog(coups_liste, cv_f[0] * np.sqrt(coups_liste[0] / np.array(coups_liste)), "k:", label="pente −1/2")
plt.xlabel("nombre de coups"); plt.ylabel("CV"); plt.legend(); plt.grid(True, which="both"); plt.show()

🔍 **Question** : que gagne-t-on avec Hann ? Que perd-on ? (revoyez les profils de l'exercice 2.2)

✍️ **Votre réponse :**



---
# Partie 4 : Reconstruction itérative MLEM

$$ x^{(k+1)} = \frac{x^{(k)}}{P^T \mathbf{1}} \; P^T\!\left( \frac{y}{P\,x^{(k)}} \right) $$

- $P$ : projecteur (image → sinogramme) ; $P^T$ : rétroprojecteur (sinogramme → image)
- $y$ : sinogramme mesuré ; $P^T\mathbf{1}$ : **sensibilité** de chaque pixel

En mots, à chaque itération : (1) on **projette** l'image courante ; (2) on **compare** aux mesures par un rapport ;
(3) on **rétroprojette** ce rapport ; (4) on **corrige** l'image en la multipliant, puis en divisant par la sensibilité.

✏️ Complétez les **deux** lignes marquées. Correspondance avec la formule :
- `P(x)` calcule $P\,x$ ; `Pt(ratio)` calcule $P^T(\text{ratio})$ ; `sens_safe` est $P^T\mathbf{1}$ ;
- multipliez aussi par `masque`, pour garder à zéro les pixels hors du champ.

In [ ]:
def P(x):
    return radon(x, theta=THETA)

def Pt(y):
    return iradon(y, theta=THETA, filter_name=None, output_size=N)

def mlem(y, n_iter=20, garder=False):
    sens = Pt(np.ones_like(y))                       # sensibilité P^T 1
    masque = sens > 1e-6 * sens.max()                # pixels vus par le détecteur
    sens_safe = np.where(masque, sens, 1.0)
    x = masque.astype(float)                         # image initiale uniforme
    hist = []
    for k in range(n_iter):
        proj  = ...                                  # ✏️ (1) projeter l'image courante x
        ratio = np.divide(y, proj, out=np.zeros_like(y), where=proj > 0)   # (2) comparer : mesuré / prédit
        x     = ...                                  # ✏️ (3) et (4) : x × Pt(ratio) / sens_safe × masque
        if garder:
            hist.append(x.copy())
    return (x, hist) if garder else x

### Exercice 4.1 : premier MLEM
Le fantôme de Shepp-Logan est acquis avec $10^6$ coups et un détecteur de FWHM 6 mm. Rien à compléter :
exécutez (environ 15 secondes), puis faites varier le nombre d'itérations avec le curseur.

In [ ]:
y_sl = acquerir(objet, 1e6, fwhm_mm=6)
x_sl, hist = mlem(y_sl, n_iter=60, garder=True)
fbp_hann = iradon(y_sl, theta=THETA, filter_name="hann")

@interact(k=IntSlider(min=1, max=60, step=1, value=10, description="itérations"))
def voir(k):
    montrer(objet, hist[k-1], fbp_hann, titres=["Objet", f"MLEM {k} it.", "FBP Hann"])

### Exercice 4.2 : semi-convergence
Pour chaque itération, on calcule :
- l'**erreur** par rapport à l'objet vrai (RMSE, après normalisation des deux images par leur somme) ;
- le **bruit** : CV dans une petite zone uniforme `roi_u`, comme dans la partie 3.

✏️ Complétez la ligne du CV (même formule qu'à l'exercice 3.1, avec l'image `xk` et la région `roi_u`).

In [ ]:
roi_u = disque(84, 88, 5) > 0                         # zone uniforme de l'objet
ref = objet / objet.sum()
rmse, cv_it = [], []
for xk in hist:
    rmse.append( np.sqrt(((xk / xk.sum() - ref)**2).mean()) )
    cv_it.append( ... )                                 # ✏️ écart-type / moyenne de xk dans roi_u
fig, ax = plt.subplots(1, 2)
ax[0].plot(range(1, 61), rmse); ax[0].set_title("Erreur vs itérations"); ax[0].set_xlabel("itérations")
ax[1].plot(range(1, 61), cv_it); ax[1].set_title("Bruit (CV) vs itérations"); ax[1].set_xlabel("itérations")
plt.show()

🔍 **Questions**
1. L'erreur décroît-elle toujours ? Pourquoi s'arrête-t-on « avant la convergence » ?
2. Quelles qualités MLEM a-t-il que la FBP n'a pas ? (pensez : positivité, modèle du bruit, modélisation physique)
3. Pourquoi les constructeurs utilisent-ils OSEM plutôt que MLEM ?

✍️ **Votre réponse :**



### ✅ Checkpoint 4
Montrez les deux courbes de l'exercice 4.2 et proposez un nombre d'itérations raisonnable.

🆘 **Cellule de secours** : si votre MLEM ne fonctionne pas, **après avoir essayé**, exécutez la cellule suivante pour pouvoir continuer.

In [ ]:
def mlem(y, n_iter=20, garder=False):
    sens = Pt(np.ones_like(y)); masque = sens > 1e-6 * sens.max(); sens_safe = np.where(masque, sens, 1.0)
    x = masque.astype(float); hist = []
    for k in range(n_iter):
        proj = P(x); ratio = np.divide(y, proj, out=np.zeros_like(y), where=proj > 0)
        x = x * Pt(ratio) / sens_safe * masque
        if garder: hist.append(x.copy())
    return (x, hist) if garder else x
print("MLEM de secours chargé.")

---
# Partie 5 : Résolution spatiale et volume partiel

L'image d'un point n'est pas un point : c'est une tache. Sa **largeur à mi-hauteur** (FWHM) mesure la résolution spatiale.

### Exercice 5.1 : à la main, sur papier
On a mesuré le profil d'une source ponctuelle (pixel de 2 mm) :

| position (mm) | −10 | −8 | −6 | −4 | −2 | 0 | 2 | 4 | 6 | 8 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| coups | 47 | 76 | 176 | 405 | 733 | 970 | 923 | 633 | 321 | 134 | 62 |

1. Tracez ce profil **à la main** sur papier millimétré, puis estimez la FWHM.
2. Le fond n'est pas nul. Faut-il le retirer avant de chercher la mi-hauteur ? Estimez-le.
3. Calculez les deux positions de mi-hauteur par **interpolation linéaire** (règle de trois entre deux points voisins).

✍️ **Votre réponse :**



Vérifiez votre calcul avec la fonction fournie ci-dessous (✏️ indiquez votre estimation du fond).

In [ ]:
x_mm = np.arange(-10, 11, 2, dtype=float)
prof = np.array([47, 76, 176, 405, 733, 970, 923, 633, 321, 134, 62], dtype=float)

def fwhm_interp(x, y, fond=0.0):
    """FWHM par interpolation linéaire entre les points de part et d'autre de la mi-hauteur."""
    y = y - fond; i_max = np.argmax(y); demi = y[i_max] / 2
    i = np.where(y[:i_max] < demi)[0][-1]                    # dernier point sous la mi-hauteur, à gauche
    x_gauche = x[i] + (demi - y[i]) * (x[i+1] - x[i]) / (y[i+1] - y[i])
    j = i_max + np.where(y[i_max:] < demi)[0][0]             # premier point sous la mi-hauteur, à droite
    x_droit = x[j-1] + (demi - y[j-1]) * (x[j] - x[j-1]) / (y[j] - y[j-1])
    return x_droit - x_gauche

plt.plot(x_mm, prof, "o-"); plt.xlabel("mm"); plt.ylabel("coups"); plt.grid(); plt.show()
print("FWHM sans fond :", round(fwhm_interp(x_mm, prof), 2), "mm")
print("FWHM avec fond :", round(fwhm_interp(x_mm, prof, fond=...), 2), "mm")   # ✏️ votre estimation du fond

### Exercice 5.2 : ajustement gaussien
On ajuste le modèle $f(x) = A\,e^{-(x-x_0)^2/2\sigma^2} + B$, puis $\text{FWHM} = 2{,}355\,\sigma$.
L'ajustement a besoin d'un **point de départ** : ✏️ lisez sur votre graphe une estimation de $A$ (hauteur au-dessus du fond),
$x_0$ (centre), $\sigma$ (environ FWHM / 2,4) et $B$ (fond), puis complétez le calcul de la FWHM.

In [ ]:
def gauss(x, A, x0, sigma, B):
    return A * np.exp(-(x - x0)**2 / (2 * sigma**2)) + B

popt, _ = curve_fit(gauss, x_mm, prof, p0=[..., ..., ..., ...])   # ✏️ [A, x0, sigma, B] lus sur le graphe
fwhm_fit = ... * popt[2]                                           # ✏️ FWHM = 2,355 × sigma
print(f"FWHM ajustée = {fwhm_fit:.2f} mm,  centre = {popt[1]:.2f} mm,  fond = {popt[3]:.0f}")
xx = np.linspace(-10, 10, 400)
plt.plot(x_mm, prof, "o"); plt.plot(xx, gauss(xx, *popt)); plt.grid(); plt.show()

🔍 **Question** : comparez les trois méthodes (lecture, interpolation, ajustement). Pourquoi la lecture « au pixel près »
est-elle insuffisante ? Combien de pixels faut-il au minimum dans la FWHM pour la mesurer correctement ?

✍️ **Votre réponse :**



### Exercice 5.3 : la résolution dépend de la reconstruction
Le détecteur a une FWHM de **8 mm**. On mesure la FWHM dans l'image reconstruite d'une source ponctuelle placée dans un fond peu actif, pour différentes
reconstructions. Rien à compléter : exécutez (environ 15 secondes) et observez.

In [ ]:
def fwhm_image(img):
    """FWHM (mm) par ajustement gaussien du profil horizontal passant par le maximum."""
    i, j = np.unravel_index(np.argmax(img), img.shape)
    p = img[i, j-12:j+13]; xs = (np.arange(p.size) - 12) * PIX
    popt, _ = curve_fit(gauss, xs, p, p0=[p.max(), 0, 3, np.median(p)])
    return 2.355 * abs(popt[2])

src = disque(64, 64, 0.5) + disque(64, 64, 50, 0.01)    # un pixel très actif, dans un fond peu actif (comme un patient)
y_src = acquerir(src, 3e6, fwhm_mm=8)
resultats = {}
resultats["FBP rampe"] = fwhm_image(iradon(y_src, theta=THETA, filter_name="ramp"))
resultats["FBP Hann"]  = fwhm_image(iradon(y_src, theta=THETA, filter_name="hann"))
_, h = mlem(y_src, n_iter=100, garder=True)
for k in [2, 5, 20, 50, 100]:
    resultats[f"MLEM {k} it."] = fwhm_image(h[k-1])
for k, v in resultats.items():
    print(f"{k:14s} : {v:5.2f} mm")

🔍 **Questions**
1. Comment évolue la FWHM avec le nombre d'itérations ? Et avec le filtre ?
2. Peut-on descendre sous 8 mm ici ? Que faudrait-il ajouter au projecteur $P$ ? (indice : « modélisation de la PSF »)
3. Pourquoi les normes (NEMA) imposent-elles la reconstruction utilisée pour mesurer la résolution ?

✍️ **Votre réponse :**



### Exercice 5.4 : le volume partiel
Un objet petit devant la résolution est « étalé » : son activité déborde sur le voisinage et sa valeur mesurée baisse.
On le chiffre par le **coefficient de recouvrement** $RC = \text{valeur mesurée} / \text{valeur vraie}$.

Le fantôme contient 6 « lésions » de 4 à 28 mm, 4 fois plus actives que le fond. Rien à compléter : exécutez (environ 15 secondes).

In [ ]:
D_mm = np.array([4, 6, 10, 14, 20, 28])        # diamètres des « lésions » (mm)
CONTRASTE = 4.0                                # lésion / fond
FWHM_DET = 8.0
fond = disque(64, 64, 55, 1.0)
angles = np.linspace(0, 2*np.pi, len(D_mm), endpoint=False)
centres = [(64 + 35*np.cos(a), 64 + 35*np.sin(a)) for a in angles]
fantome = fond.copy()
for (cx, cy), d in zip(centres, D_mm):
    fantome[disque(cx, cy, d / 2 / PIX) > 0] = CONTRASTE
roi_fond = disque(64, 64, 12) > 0

def mesurer_rc(img):
    """RC max (maximum autour de la lésion) et RC moyen (moyenne dans le diamètre vrai), fond normalisé à 1."""
    img = img / img[roi_fond].mean()
    rc_max, rc_moy = [], []
    for (cx, cy), d in zip(centres, D_mm):
        r = d / 2 / PIX
        rc_max.append(img[disque(cx, cy, r + 2) > 0].max() / CONTRASTE)
        rc_moy.append(img[disque(cx, cy, max(r, 0.5)) > 0].mean() / CONTRASTE)
    return np.array(rc_max), np.array(rc_moy)

y_ph = acquerir(fantome, 2e7, fwhm_mm=FWHM_DET)
rec_ph = mlem(y_ph, n_iter=50)
montrer(fantome, rec_ph, titres=["Vrai", "MLEM 50 it."])
rcM, rcm = mesurer_rc(rec_ph)
plt.plot(D_mm / FWHM_DET, rcM, "o-", label="RC max")
plt.plot(D_mm / FWHM_DET, rcm, "s-", label="RC moyen")
plt.axhline(1, color="k", ls="--"); plt.xlabel("diamètre / FWHM"); plt.ylabel("RC")
plt.legend(); plt.grid(); plt.show()
for d, a, b in zip(D_mm, rcM, rcm):
    print(f"{d:2d} mm : RC max = {a:.2f}   RC moyen = {b:.2f}")

🔍 **Questions**
1. À partir de quel rapport diamètre / FWHM le RC max dépasse-t-il 0,9 ?
2. Pourquoi RC moyen < RC max ?
3. Ici les objets sont des **disques** (2D). En 3D, pour des sphères, l'effet serait-il plus ou moins marqué ?

✍️ **Votre réponse :**



### Exercice 5.5 : conséquence clinique (sur papier)
Une lésion de 10 mm a un SUVmax vrai de 8. La TEP a une résolution reconstruite de 8 mm.
1. À l'aide de votre courbe, quel SUVmax va-t-on mesurer ?
2. Sous traitement, la lésion passe à 6 mm **sans changer de fixation par cellule**. Qu'affiche la TEP ? Quel risque d'interprétation ?
3. Citez deux moyens de limiter ou de corriger l'effet de volume partiel.

✍️ **Votre réponse :**



### ✅ Checkpoint 5 (final)
La courbe RC = f(diamètre / FWHM) commentée, et la réponse argumentée à l'exercice 5.5.

---
# ⭐ Bonus (pour ceux qui ont fini)

### Bonus A : le volume partiel dépend de la reconstruction
✏️ Complétez la liste pour superposer les courbes de RC max pour MLEM à 10 et 50 itérations, et pour la FBP Hann.

In [ ]:
images = {
    "MLEM 10 it.": mlem(y_ph, n_iter=10),
    "MLEM 50 it.": rec_ph,
    "FBP Hann":    iradon(y_ph, theta=THETA, filter_name=...),   # ✏️
}
for nom, img in images.items():
    plt.plot(D_mm / FWHM_DET, mesurer_rc(img)[0], "o-", label=nom)
plt.axhline(1, color="k", ls="--"); plt.xlabel("diamètre / FWHM"); plt.ylabel("RC max"); plt.legend(); plt.grid(); plt.show()

✍️ **Votre réponse :**



### Bonus B : bruit et SUVmax
Même fantôme, avec seulement $2\cdot10^6$ coups, 5 réalisations du bruit. Rien à compléter : exécutez (environ 30 secondes).

In [ ]:
res = np.array([mesurer_rc(mlem(acquerir(fantome, 2e6, fwhm_mm=FWHM_DET), n_iter=50)) for _ in range(5)])
for k, nom in enumerate(["RC max", "RC moyen"]):
    plt.errorbar(D_mm / FWHM_DET, res[:, k].mean(0), yerr=res[:, k].std(0), marker="o", capsize=3, label=nom + " (2·10⁶ coups)")
plt.plot(D_mm / FWHM_DET, rcM, "k:", label="RC max (2·10⁷ coups)")
plt.axhline(1, color="k", ls="--"); plt.xlabel("diamètre / FWHM"); plt.ylabel("RC"); plt.legend(); plt.grid(); plt.show()

🔍 **Questions**
1. Le RC max peut-il dépasser 1 ? Pourquoi le SUVmax est-il **biaisé par le bruit** ?
2. Pourquoi a-t-on proposé le SUVpeak (moyenne dans une sphère de 1 cm³ autour du maximum) ?

✍️ **Votre réponse :**



### Bonus C : écrire soi-même l'interpolation
Relisez la fonction `fwhm_interp` de l'exercice 5.1 et commentez chaque ligne avec vos mots. Que se passe-t-il si le profil
ne redescend jamais sous la mi-hauteur d'un côté ?

✍️ **Votre réponse :**



---
**Avant de partir** : *File → Download* (ou *Commit & Push*), puis **arrêtez votre codespace**.